# Task 2: Self-Attention Mechanism

Self-attention is the fundamental building block of transformers. It allows each position in a sequence to attend to all positions in the previous layer.

## What You'll Learn
1. **Query, Key, Value** - The three vectors used in attention
2. **Scaled Dot-Product Attention** - The core attention computation
3. **Attention Weights** - How to visualize what the model attends to
4. **Why Scaling** - Understanding the scaling factor

## Theory: How Self-Attention Works

Given an input sequence, self-attention computes:

**Attention(Q, K, V) = softmax(QK^T / sqrt(d_k))V**

Where:
- **Q (Query)**: What I'm looking for
- **K (Key)**: What I can offer
- **V (Value)**: What I actually give
- **d_k**: Dimension of keys (for scaling)

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

## 1. Creating Q, K, V from Input

In [ ]:
# Example: sequence length = 4, batch size = 1, embedding dim = 8
seq_len = 4
batch_size = 1
embed_dim = 8

# Simulate input embeddings (e.g., word embeddings)
x = torch.randn(batch_size, seq_len, embed_dim)
print(f"Input shape: {x.shape}  (batch, seq_len, embed_dim)")

# Create linear projections to get Q, K, V
# In practice, we use the same embedding dim for all
d_k = embed_dim  # dimension for keys (also used for queries)
d_v = embed_dim  # dimension for values

W_q = torch.randn(embed_dim, d_k)
W_k = torch.randn(embed_dim, d_k)
W_v = torch.randn(embed_dim, d_v)

# Project input to Q, K, V
Q = x @ W_q
K = x @ W_k
V = x @ W_v

print(f"Q shape: {Q.shape}")
print(f"K shape: {K.shape}")
print(f"V shape: {V.shape}")

## 2. Scaled Dot-Product Attention

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Compute scaled dot-product attention.
    
    Args:
        Q: (batch, seq_len_q, d_k)
        K: (batch, seq_len_k, d_k)
        V: (batch, seq_len_k, d_v)
        mask: Optional mask (batch, seq_len_q, seq_len_k)
    
    Returns:
        output: (batch, seq_len_q, d_v)
        attention_weights: (batch, seq_len_q, seq_len_k)
    """
    d_k = Q.size(-1)
    
    # Compute attention scores: Q @ K^T
    scores = torch.matmul(Q, K.transpose(-2, -1))
    
    # Scale by sqrt(d_k)
    scaled_scores = scores / np.sqrt(d_k)
    
    # Apply mask if provided
    if mask is not None:
        scaled_scores = scaled_scores.masked_fill(mask == 0, -1e9)
    
    # Softmax to get attention weights
    attention_weights = F.softmax(scaled_scores, dim=-1)
    
    # Apply attention to values
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights

# Test the attention function
output, attn_weights = scaled_dot_product_attention(Q, K, V)
print(f"Attention output shape: {output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")
print(f"\nAttention weights (should sum to 1 per row):")
print(attn_weights[0])  # First batch element

In [ ]:
# Visualize attention weights
def plot_attention(attention_weights, title="Attention Weights"):
    """Plot attention heatmap."""
    plt.figure(figsize=(6, 5))
    plt.imshow(attention_weights, cmap='viridis', aspect='auto')
    plt.colorbar()
    plt.title(title)
    plt.xlabel("Key position")
    plt.ylabel("Query position")
    plt.tight_layout()
    plt.show()

# Plot attention for our example
plot_attention(attn_weights[0].detach().numpy(), "Self-Attention Weights")

## 3. Why Scale? The Gradient Problem

In [ ]:
# Let's see why scaling matters
d_k_large = 64  # Typical BERT dimension

Q_large = torch.randn(1, 10, d_k_large)
K_large = torch.randn(1, 10, d_k_large)

# Without scaling - dot products can be large
scores_unscaled = torch.matmul(Q_large, K_large.transpose(-2, -1))
print(f"Unscaled scores - mean: {scores_unscaled.mean().item():.2f}, std: {scores_unscaled.std().item():.2f}")
print(f"Unscaled - min: {scores_unscaled.min().item():.2f}, max: {scores_unscaled.max().item():.2f}")

# After softmax, this concentrates too much on one value
weights_unscaled = F.softmax(scores_unscaled, dim=-1)
print(f"\nUnscaled softmax - max weight: {weights_unscaled.max().item():.4f}")

# With scaling
scores_scaled = scores_unscaled / np.sqrt(d_k_large)
weights_scaled = F.softmax(scores_scaled, dim=-1)
print(f"\nScaled softmax - max weight: {weights_scaled.max().item():.4f}")

print("\n→ With scaling, attention is more evenly distributed!")

## 4. Self-Attention as a PyTorch Module

In [ ]:
class SelfAttention(torch.nn.Module):
    """Self-attention module with learned projections."""
    
    def __init__(self, embed_dim, d_k=None, d_v=None):
        super().__init__()
        self.embed_dim = embed_dim
        self.d_k = d_k if d_k is not None else embed_dim
        self.d_v = d_v if d_v is not None else embed_dim
        
        # Linear projections for Q, K, V
        self.W_q = torch.nn.Linear(embed_dim, self.d_k)
        self.W_k = torch.nn.Linear(embed_dim, self.d_k)
        self.W_v = torch.nn.Linear(embed_dim, self.d_v)
        
    def forward(self, x, mask=None):
        """
        Args:
            x: (batch, seq_len, embed_dim)
            mask: Optional attention mask
        Returns:
            output: (batch, seq_len, d_v)
            attention_weights: (batch, seq_len, seq_len)
        """
        batch_size, seq_len, _ = x.shape
        
        # Project to Q, K, V
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        
        # Compute attention
        output, attention_weights = scaled_dot_product_attention(Q, K, V, mask)
        
        return output, attention_weights

# Test the module
attention = SelfAttention(embed_dim=8, d_k=8, d_v=8)
x = torch.randn(2, 4, 8)  # batch=2, seq_len=4, embed_dim=8
output, weights = attention(x)

print(f"Input: {x.shape}")
print(f"Output: {output.shape}")
print(f"Attention weights: {weights.shape}")

In [ ]:
# Verify that attention can learn (backpropagation works)
attention = SelfAttention(embed_dim=16, d_k=16, d_v=16)
optimizer = torch.optim.Adam(attention.parameters(), lr=0.01)

# Random "target" we want the model to produce
x = torch.randn(1, 5, 16)
target = torch.randn(1, 5, 16)

for i in range(50):
    output, _ = attention(x)
    loss = F.mse_loss(output, target)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (i + 1) % 10 == 0:
        print(f"Step {i+1}, Loss: {loss.item():.4f}")

print("\n✓ Attention module can learn via backprop!")

## Exercises

### Exercise 1: Implement Self-Attention from Scratch
Implement self-attention WITHOUT using any PyTorch operations except basic matrix ops (matmul, transpose). Use only `torch.matmul`, `torch.transpose`, `torch.softmax`, and basic math operations.

In [ ]:
# SOLUTION
def self_attention_scratch(x, W_q, W_k, W_v):
    """Self-attention from complete scratch."""
    d_k = W_q.shape[0]
    
    # Project to Q, K, V
    Q = torch.matmul(x, W_q)
    K = torch.matmul(x, W_k)
    V = torch.matmul(x, W_v)
    
    # Attention scores: Q @ K^T
    scores = torch.matmul(Q, torch.transpose(K, -2, -1))
    
    # Scale
    scaled = scores / (d_k ** 0.5)
    
    # Softmax
    attn_weights = torch.softmax(scaled, dim=-1)
    
    # Apply to V
    output = torch.matmul(attn_weights, V)
    
    return output, attn_weights

# Test
x = torch.randn(1, 4, 8)
W_q, W_k, W_v = [torch.randn(8, 8) for _ in range(3)]
out, w = self_attention_scratch(x, W_q, W_k, W_v)
print(f"Output shape: {out.shape}, Weights shape: {w.shape}")

### Exercise 2: Masked Attention
Implement attention with a causal mask (so each position can only attend to previous positions). This is used in decoder-only models like GPT.

In [ ]:
# SOLUTION
def create_causal_mask(seq_len):
    """Create a causal mask (lower triangular)."""
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask

def masked_attention(Q, K, V):
    """Attention with causal masking."""
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (d_k ** 0.5)
    
    # Apply causal mask
    mask = create_causal_mask(scores.size(-1))
    scores = scores.masked_fill(mask == 0, -1e9)
    
    weights = torch.softmax(scores, dim=-1)
    output = torch.matmul(weights, V)
    return output, weights

# Test
Q = K = V = torch.randn(1, 4, 8)
out, weights = masked_attention(Q, K, V)

print("Causal mask applied:")
print(weights[0])  # Lower triangular

### Exercise 3: Multi-Token Attention Visualization
Create a longer sequence (e.g., 10 tokens) and visualize how attention learns to attend to different positions. Use different random seeds and observe patterns.

In [ ]:
# SOLUTION
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, seed in enumerate([42, 123, 456]):
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    # Create new attention module with this seed
    attn = SelfAttention(embed_dim=32, d_k=32, d_v=32)
    
    # Random input
    x = torch.randn(1, 10, 32)
    
    # Get attention weights
    _, weights = attn(x)
    
    # Plot
    axes[idx].imshow(weights[0].detach().numpy(), cmap='viridis')
    axes[idx].set_title(f'Seed {seed}')
    axes[idx].set_xlabel('Key position')
    axes[idx].set_ylabel('Query position')

plt.suptitle('Attention Patterns with Different Initializations', y=1.02)
plt.tight_layout()
plt.show()

## Summary

You learned:
- ✅ Query, Key, Value concept and projections
- ✅ Scaled dot-product attention computation
- ✅ Why scaling by sqrt(d_k) is important
- ✅ How to implement self-attention as a PyTorch module
- ✅ Visualizing attention weights

## Next Task
[Task 3: Multi-Head Attention](../task-03-multi-head-attention/overview.html) - We'll extend self-attention to multiple heads!